# SMS Spam Detection

**Goal:** Classify SMS messages as Spam or Ham (legitimate)
**Algorithm:** Naive Bayes with TF-IDF (best for text classification)
**Dataset:** [SMS Spam Collection](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score)
%matplotlib inline

In [ ]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


## 1. Load Data from Kaggle

In [ ]:
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
df = pd.read_csv(f"{path}/spam.csv", encoding='latin-1')

# Keep only needed columns
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'message'})
print ('Shape: %s' % (df.shape,))
print ('First 3 rows:\n%s' % df.head(3))

<hr>## 2. Exploratory Data Analysis

In [ ]:
print ('Label distribution:\n%s' % df['label'].value_counts())
spam_ratio = df['label'].value_counts(normalize=True)['spam'] * 100
print ('\nSpam ratio: %.2f%%' % spam_ratio)
print ('\nMessage length stats:')
df['length'] = df['message'].apply(len)
print (df.groupby('label')['length'].describe())

In [ ]:
# Visualize message lengths by class
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df[df['label']=='ham']['length'].hist(bins=50, alpha=0.7, label='Ham', color='blue')
df[df['label']=='spam']['length'].hist(bins=50, alpha=0.7, label='Spam', color='red')
plt.xlabel('Message Length')
plt.ylabel('Count')
plt.legend()
plt.title('Message Length Distribution')

plt.subplot(1, 2, 2)
df['label'].value_counts().plot(kind='pie', autopct='%.1f%%', colors=['blue', 'red'])
plt.title('Spam vs Ham')
plt.tight_layout()
plt.show()

<hr>## 3. Text Preprocessing

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['message'].apply(clean_text)
print ('Sample ham:', df[df['label']=='ham']['message'].iloc[0][:60])
print ('Sample spam:', df[df['label']=='spam']['message'].iloc[0][:60])

<hr>## 4. Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X = vectorizer.fit_transform(df['clean']).toarray()
y = df['label'].map({'ham': 0, 'spam': 1})

print ('Feature matrix: %s' % (X.shape,))
print ('Vocabulary size: %d words' % len(vectorizer.get_feature_names_out()))

<hr>## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print ('Train: %d, Test: %d' % (X_train.shape[0], X_test.shape[0]))

<hr>## 6. Train Model

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)
print ('Model: %s' % model)

<hr>## 7. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)

print ('Accuracy:  %.4f' % accuracy_score(y_test, y_pred))
print ('Precision: %.4f' % precision_score(y_test, y_pred))
print ('Recall:    %.4f' % recall_score(y_test, y_pred))
print ('\nConfusion Matrix:\n%s' % confusion_matrix(y_test, y_pred))
print ('\nClassification Report:')
print (classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

<hr>## 8. Test Custom Messages

In [ ]:
test_messages = [
    'Hey, how are you? Want to grab coffee tomorrow?',
    'CONGRATULATIONS! You have won $1000 cash prize! Call now!'
]

print ('Custom message predictions:')
for msg in test_messages:
    cleaned = clean_text(msg)
    vec = vectorizer.transform([cleaned]).toarray()
    prob = model.predict_proba(vec)[0]
    pred = model.predict(vec)[0]
    label = 'SPAM' if pred == 1 else 'HAM (Safe)'
    print ("  '%s'" % msg[:40])
    print ("  -> %s (spam confidence: %.1f%%)" % (label, prob[1]*100))
    print ()